# BEE 4750 Homework 2: Systems Modeling and Simulation

**Name**:

**ID**:

> **Due Date**
>
> Thursday, 09/25/25, 9:00pm

## Overview

### Instructions

-   Problem 1 asks you to draw a systems diagram and identify the type
    of a feedback.
-   Problem 2 asks you to model contaminant concentrations in a river
    and use simulation to compare the concentrations to a regulatory
    standard.
-   Problem 3 asks you to explore the implications of an ice-albedo
    feedback in the Earth’s climate system by understanding the
    equilibria of the modeled system and their stabilities.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/BEE_4750/hw2-natalie`
   Installed JpegTurbo_jll ─────────────── v3.1.3+0
   Installed Libmount_jll ──────────────── v2.41.1+0
   Installed InvertedIndices ───────────── v1.3.1
   Installed InlineStrings ─────────────── v1.4.5
   Installed Accessors ─────────────────── v0.1.42
   Installed Roots ─────────────────────── v2.2.10
   Installed Preferences ───────────────── v1.5.0
   Installed DataFrames ────────────────── v1.8.0
   Installed Fontconfig_jll ────────────── v2.17.1+0
   Installed WorkerUtilities ───────────── v1.6.1
   Installed SentinelArrays ────────────── v1.4.8
   Installed WeakRefStrings ────────────── v1.4.2
   Installed DataStructures ────────────── v0.19.1
   Installed StatsBase ─────────────────── v0.34.6
   Installed Plots ─────────────────────── v1.40.19
   Installed PrettyTables ──────────────── v3.0.8
   Installed TableTraits ───────────────── v1.0.1
   Installed Expat_jll ─────────────────── v2.7.1+0
   Installed PooledArrays ─────────

In [2]:
using Plots
using LaTeXStrings
using CSV
using DataFrames
using Roots

## Problems (Total: 30 Points)

### Problem 1 (5 points)

Draw a systems diagram for the relationship between global mean
temperature, atmospheric CO<sub>2</sub> concentrations, and ocean
CO<sub>2</sub> concentrations. What are the signs of the interactions
between these components and why? What does this suggest about the
overall feedback between temperature and the ocean carbon cycle?

> **Tip**
>
> Think about Henry’s law for CO<sub>2</sub>.

### Problem 2 (15 points)

A river which flows at 10 km/d is receiving discharges of wastewater
contaminated with CRUD from two sources which are 15 km apart, as shown
in the Figure below. CRUD decays exponentially in the river at a rate of
0.36 $\mathrm{d}^{-1}$ and is deposited by the atmosphere along the
river at a rate of 54 kg/km/d. 

<figure>
<img src="attachment:figures/river_diagram.png"
alt="Schematic of the river system in Problem 2" />
<figcaption aria-hidden="true">Schematic of the river system in Problem
2</figcaption>
</figure>

#### Problem 2.1

Draw a systems diagram with the relevant control volume(s) denoted and
any relevant in/out-flows between these boxes. How did you decide how
many boxes were needed?

We decided to use two control volumes: the first CV from the first discharge from the upstream river, and the second CV from the second discharge (15 km away the first discharge point). These two control volumes are the "junctions" of the diagram, in that they have some input and output.

$$
\text{[ River Inflow ]}
\;\;\longrightarrow\;\;
\boxed{\text{CV1: Discharge 1 + Deposition + Decay}}
\;\;\longrightarrow\;\;
\boxed{\text{CV2: Discharge 2 + Deposition + Decay}}
\;\;\longrightarrow\;\;
\text{[ River Outflow ]}
$$


#### Problem 2.2

Develop a model for the concentration of CRUD downriver by formulating
and solving the appropriate differential equation(s) analytically.

> **Tip**
>
> Formulate your model in terms of distance downriver, rather than
> leaving it in terms of time from discharge.

We want to model the concentration of CRUD, $C(x)$, as a function of distance downriver.
Two parameters that we are given are the $velocity$ and $k_{rate}$. We can find the first-order decay rate as $k_x$:

$$
k_x = \frac{k_t}{v} = \frac{0.36}{10} = 0.036 \;\text{km}^{-1}.
$$

As seen fomr the systems diagram, the fluctuating values are deposition and decay.
Using mass balancing, we need to account for those variations by using the following ODE model:

$$
\frac{dy}{dx} = -Decay Rate + Deposition Rate.
$$

Substituting in our relevant variables gives us:

$$
\frac{dC}{dx} = -k_x C + \frac{D}{Q/100},
$$

where we are given all the parameters. $C(x)$ = CRUD concentration (kg per 1000 m$^3$),  $k_x$ = decay constant per km,  $D = 54 \;\text{kg/km/day}$ = atmospheric deposition rate,  and $Q$ = river flow (m$^3$/day).

For the first CV, our equation is 

$$
\frac{dC_1}{dx} = -k_x C_1 + \frac{D}{Q/100},

C_1(0) = C_0
$$.

For the second CV, our equation is 

$$
\frac{dC_2}{dx} = -k_x C + \frac{D}{Q/100},
C_2(15) = C_1(15) + \frac{Q_2}{Q}
$$.

Solving analytically, the solution to this first-order linear ODEs are

$$
C(x_1) = \left(C(0) - \frac{D}{Q k_x}\right) e^{-k_x x} + \frac{D}{Q k_x}.
$$

and 

$$
C(x_2) = \left(C_2(15) - \frac{D}{Q k_x}\right) e^{-k_x (x-15)} + \frac{D}{Q k_x}.
$$

The first term, $\left(C(0) - \tfrac{D}{Qk_x}\right)e^{-k_x x}$, represents the decay factor, while the second term, $\tfrac{D}{Qk_x}$, represents the deposition


In [39]:
# given parameters
# river inflow
Q0 = 250_000.0         # (m^3/day)
C0 = 0.5               # (kg/1000 m^3)

dist = 15 # km
v = 10.0               # km/day
k = 0.36          # 1/day
k_rate = k_time / v   # per km
deposition_rate = 54.0        # kg/km/day
x2 = 15.0

# crud concentration should be a function of distance downriver (x)

# first CV
function CRUD_conc1(x, Q, Cinit)
    deposition_factor = deposition_rate / Q / k_rate
    return (Cinit - deposition_factor) * exp(-k_rate * x) + deposition_factor
end

# second CV
function CRUD_conc2(x, Q, C_from1, Q2, x2)
    deposition_factor = deposition_rate / Q / k_rate
    C_addition = C_from1 + Q2 / Q
    return (C_addition - deposition_factor) * exp(-k_rate * (x - x2)) + deposition_factor
end


CRUD_conc2 (generic function with 2 methods)

#### Problem 2.3

Determine if the system in compliance with a regulatory limit of
$2.3\ \text{kg}/(1000 \text{m}^3)$. You can do this analytically or
computationally.

In [40]:
C_limit = 2.3 # kg/(1000m^3)

# first discharge
Q1 = (Q0 + 40_000)
C1 = ((Q0*C0 + 40_000*9) / Q1) 
C15 = CRUD_conc1(15, Q1, C1)


# second discharge 
Q2 = Q1 + 60_000
C2 = (Q1*C15 + 60_000*7) / Q2
C_result = CRUD_conc2(25, Q2, C15, 60_000, x2) 

println("Concentration after first 15 km: $C15")
println("Concentration at end (25 km): $Cend")

if C_result <= C_limit
    println("Yes, system is in compliance with limit")
else
    println("No, system is NOT in compliance with limit")
end


Concentration after first 15 km: 0.976754413871807
Concentration at end (25 km): 0.8023557597299038
Yes, system is in compliance with limit


### Problem 3 (10 points)

In class, we discussed the ice-albedo feedback and its possible
influence on melting the hypothesized [“Snowball
Earth”](https://en.wikipedia.org/wiki/Snowball_Earth). In this problem,
we’ll introduce a simple model of the Earth’s energy balance with
includes this feedback and examine the stability of the climate.

This simple model of the energy balance (averaged over the entire
planet) is:

<span id="eq-climate">$$
\underbrace{C\frac{dT}{dt}}_{\text{change in heat}} = \underbrace{\frac{(1-\alpha)S}{4}}_{\text{incoming radiation}} - \underbrace{(A - BT)}_{\text{outgoing radiation}} + \underbrace{a\ln \left(\frac{[CO_2]}{[CO_2]_{PI}}\right)}_{\text{greenhouse effect}},
 \qquad(1)$$</span>

where:

-   $T$ is the Earth’s global mean temperature (in $^\circ\text{C}$);
-   $C$ is the heat capacity of the atmosphere and shallow ocean, taken
    to be $51\ \text{J}/\text{m}^2/^\circ\text{C}$;
-   $\alpha$ is the planetary albedo, or the fraction of incoming
    radiation reflected by the Earth, which has a present-day value of
    approximately 0.3;
-   $S$ is the solar constant, or the amount of solar radiation received
    by the Earth averaged over area, which has a present-day value of
    $1368\ \text{W}/\text{m}^2$ but during the Neoproteorozoic Era had a
    value of $1272\ \text{W}/\text{m}^2$ (this is divided by four
    because the Earth is a sphere but $S$ is the radiation captured by a
    disc with radius equal to that of the Earth);
-   $A$ and $B$ are coefficients from the linearization of outgoing
    radiation physics, and have corresponding values
    $B=-1.3\ \text{W}/\text{m}^2/^\circ\text{C}$ (estimated from a
    number of lines of evidence about the sensitivity of outgoing
    radiation to temperature; this is negative due to a sign convention
    about the direction of incoming vs. outgoing radiation) and
    $A=221.2\ \text{W}/\text{m}^2$ (estimated by assuming the
    pre-industrial temperature of $14^\circ\text{C}$ was stable without
    anthropogenic greenhouse gas emissions).

We will ignore the greenhouse effect term as we are considering the
Earth system well before humans were around.

#### Problem 3.1

Discretize the climate model
(<a href="#eq-climate" class="quarto-xref">Equation 1</a>) using forward
Euler integration and a time step of $\Delta t = 1$ yr.

#### Problem 3.2

Rather than assuming a constant value for the albedo $\alpha$, we will
represent the ice-albedo feedback by letting $\alpha$ depend on $T$:

$$\alpha(T) = 
    \begin{cases} 
        \alpha_i & \quad \text{if } T \leq -10^\circ \text{C} \\
        \alpha_i + (\alpha_0 - \alpha_i)((T+10) / 20) & \quad \text{if } -10^\circ \text{C} \leq T \leq 10^\circ \text{C} \\
        \alpha_0 & \quad \text{if } T \geq 10^\circ \text{C}.
    \end{cases}$$

Let $\alpha_i = 0.5$ and $\alpha_0 = 0.3$. Simulate the simple climate
model using the Neoprotereozoic value of $S$ with temperature-varying
albedo for initial values of $T$ spanning
$-60^\circ\text{C} \leq T_0 \leq 30^\circ\text{C}$ over a period of 200
years. Plot the temperature trajectories. How many equilibria are there
and what are their stabilities?

#### Problem 3.3

One might hypothesize that one cause for the transition from Snowball
Earth (the stable frozen equilibrium from Problem 3.2) was an increasing
amount of incoming solar radiation (as expressed by an increase in $S$).
Examine this hypothesis using our simple model. Is this factor enough to
cause the planet to warm to the pre-industrial temperature of
$14^\circ\text{C}$?

## References

List any external references consulted, including classmates.